In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [10]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [11]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [12]:
assistant.rag('how do i get started with the course?')

'To get started with the course, follow these steps:\n\n1. **Review the core resources:**\n   * [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n   * [General Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n   * [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n\nYou can start at any time. The videos and GitHub materials are available, and deadlines are listed on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\n2. **Follow the typical weekly workflow:**\n   1. Watch the lesson videos.\n   2. Work through the lesson notebooks/code.\n   3. Read the homework instructions on GitHub.\n   4. Submit your answers through the course platform before the deadline.'

In [18]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model="gemini-3.6-flash",
    messages=messages,

)

# Extract and print text content
print(response.choices[0].message.content)

I’d love to help you, but I need a little more context! Since I’m an AI assistant, I don't know which specific course you are referring to. 

Could you tell me:
1. **What is the name of the course?**
2. **Where did you find it?** (e.g., Coursera, Udemy, YouTube, a specific university, or a creator's website)

Once you share those details, I can give you instructions on how to enroll or help you find the right place to sign up!


In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [20]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )



In [ ]:
search('how do i run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [14]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [22]:
response = openai_client.chat.completions.create(
    model="gemini-3.6-flash",
    messages=messages,
    tools=[search_tool],
)

# 3. Access and print output text
print(response.choices[0].message.content)

None


In [23]:
len(response.choices[0].message.tool_calls)

1

In [24]:
call = response.choices[0].message.tool_calls[0]

In [26]:
call


ChatCompletionMessageFunctionToolCall(id='call_1131528', function=Function(arguments='{"query":"can I join the course"}', name='search'), type='function', extra_content={'google': {'thought_signature': 'Er0FCroFAWkUfRO/lDgDvwDHSZi7ZsdaabT9JyFdx/h82vSIenY9yEFtpffASt19xq4+lEQ7/x11FUDu9QCj3ZdNDSCfuG6GAqM/DqJKn0ZvBI/jZo2WF8+kTpkgph1/aeavOjscTjvqEQUpYdTXG5qZgs2bK8X+dunGRRziwsHp9cwOSyRrQ0xaCJj2fWmCaK8GMqLJjksn8ba0HlGZttiNzdBSncMkYysVRQIXE4/ln5W9ujbwVpj6JcQiPGuPH2/+cE4pj1B1hmk7Ha+mneaaHAssEgt5IhxfFXmrlpX+j6590TRJ9By+L8Kqu6YV++ZzYkT9alpQCaY1aZwQgiiUSd2drfn3/JOz+0/SjZJ3V1TTnAjrM4GS8cVKt3yfC13lIiagYUvGescYP3Ay1WXt9ID+ltbmoJUXXil+t+Y7kzD64prcu+dLk78JruxgvBEjQRKloMDFunv9BI4F3UsTtGm1y1GtpkA1+CvGIGridLQH41wAYPZ9HA+6NnU8fhSEsb7N5ttgOp70ho1sip/2VrCPsWNJF6t6J4CkAzhPqgixU95VuAePFjli5lFdxyfYnbxxovvJd7VdcNVx6xI41BZdumW9cZvUS9My/y8hVpmRrnx5ZGQuSTu0ZWQAF3NFJxlCk2RnaXF1yUSkh/uSoHV47qC541d+frK6/dID4IhzIoqM7g/e7KG70lZxwkI9qj8G7sOp8pf90eozwrRcVqsCxH9Abm5ZiiUK3q2DByZs9DNhe2tyEF0qH4iGXJlusaLH/ksDoeFLIlIXA7EOZSt0y

In [27]:
args = call.function.arguments
args

'{"query":"can I join the course"}'

In [28]:
call.function.name

'search'

In [29]:
results = search(args)

In [30]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through th

In [32]:
import json

results_json = json.dumps(results, indent=2)

In [33]:
function_call_output = {
    "role": "tool",
    "tool_call_id": call.id,
    "content": results_json
}

In [34]:
messages.append(response.choices[0].message)

In [40]:
messages.append(function_call_output)

In [41]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_1131528', function=Function(arguments='{"query":"can I join the course"}', name='search'), type='function', extra_content={'google': {'thought_signature': 'Er0FCroFAWkUfRO/lDgDvwDHSZi7ZsdaabT9JyFdx/h82vSIenY9yEFtpffASt19xq4+lEQ7/x11FUDu9QCj3ZdNDSCfuG6GAqM/DqJKn0ZvBI/jZo2WF8+kTpkgph1/aeavOjscTjvqEQUpYdTXG5qZgs2bK8X+dunGRRziwsHp9cwOSyRrQ0xaCJj2fWmCaK8GMqLJjksn8ba0HlGZttiNzdBSncMkYysVRQIXE4/ln5W9ujbwVpj6JcQiPGuPH2/+cE4pj1B1hmk7Ha+mneaaHAssEgt5IhxfFXmrlpX+j6590TRJ9By+L8Kqu6YV++ZzYkT9alpQCaY1aZwQgiiUSd2drfn3/JOz+0/SjZJ3V1TTnAjrM4GS8cVKt3yfC13lIiagYUvGescYP3Ay1WXt9ID+ltbmoJUXXil+t+Y7kzD64prcu+dLk78JruxgvBEjQRKloMDFunv9BI4F3UsTtGm1y1GtpkA1+CvGIGridLQH41wAYPZ9HA+6NnU8fhSEsb7N5ttgOp70ho1sip/2VrCPsWNJF6t6J4CkAzhPqgixU95VuAePFjli5lFdxyfYnbxxovvJ

In [36]:
final_response = openai_client.chat.completions.create(
    model="gemini-3.6-flash",
    messages=messages
)

# 4. Print the final answer
print(final_response.choices[0].message.content)

Yes, you can still join! 

However, if you want to receive a certificate, you will need to submit your project while project submissions are still open for the current cohort.


In [37]:
usage = response.usage
usage.input_tokens, usage.output_tokens

AttributeError: 'CompletionUsage' object has no attribute 'input_tokens'

In [ ]:
# Extract the usage object
usage = response.usage

# Access token counts
input_tokens = usage.prompt_tokens
output_tokens = usage.completion_tokens

print(f"Input Tokens: {input_tokens}, Output Tokens: {output_tokens}")

Input Tokens: 74, Output Tokens: 21


In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [11]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [39]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [12]:
import json

question = "I just discovered the course. Can I join it?"

# 1. Use "system" instead of "developer" for Gemini compatibility
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]



In [16]:
response = openai_client.chat.completions.create(
    model="gemini-3.6-flash",
    messages=messages,
    tools=[search_tool],
)

In [19]:
assistant_message = response.choices[0].message


In [20]:
assistant_message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_84620', function=Function(arguments='{"query":"join course deadline late discovered"}', name='search'), type='function', extra_content={'google': {'thought_signature': 'EuECCt4CAWkUfRN/d9YHWZoMHtxuPK8UVcTjmKr+rAzXVm5r4Ed6DvNzumYJBV2WvaGeiPsVmGaa8TiqxwshoksrOE6VSQ2A/Xc2DZb48axMjJVyCqnxAgIT19iKLFFb821TZIkrotrZIwKqcIcZLJnb56iSu40pW+XTmZdYdUNjNMnQ9ZcROLBAdWBJe9J0uwZs1ZrZYbSMHXtiLjbWwflQ3TFgScGLHsoOmz2BkWToNxSc6d0axmGXckjlTadASm9/Tvc2x+3q3U235rBtbI8n9X5zvvTXis7ANUSBoooJXwYaD6ZlT/DGekHljAugdVgr7S4n20jN4r7e8gKUM3cEJFKImNAJepyTLhcaziEIpV40I1soYUDAnHfcvwzQE3hkycyNhOG3V5XCmFV+Hz2Zc1sjwmRBpy0i/M4kHEGcfPyCGwk1T0eNqDul03VDvyggLhisE5LghYviBJbgeh/9HwI='}})])